# Carga Full e Incremental

- ***Carga Full***: substitui os dados do destino por uma cópia completa da origem.
- ***Carga Incremental***: identificar os registros novos ou alterados e aplica as mudanças no destino.

## 1. Preparação

In [7]:
import pandas as pd
from pathlib import Path

PASTA = Path("data")
PASTA.mkdir(exist_ok=True)

ARQUIVO_ORIGEM = PASTA / "pedidos_origem.xlsx"
ARQUIVO_DESTINO = PASTA / "pedidos_destino.xlsx"

# Criando os registros na origem
pedidos = pd.DataFrame([
    {"id_pedido": 1, "produto": "Hamburguer", "quantidade": 2, "valor_total": 50.00, "data_atualizacao": "2026-11-18 09:00:00"},
    {"id_pedido": 2, "produto": "Pizza", "quantidade": 1, "valor_total": 40.00, "data_atualizacao": "2026-11-18 09:05:00"},
    {"id_pedido": 3, "produto": "Refrigerante", "quantidade": 3, "valor_total": 30.00, "data_atualizacao": "2026-11-18 09:10:00"}
])

pedidos["data_atualizacao"] = pd.to_datetime(pedidos["data_atualizacao"])
pedidos.to_excel(ARQUIVO_ORIGEM, index=False)

print("Arquivo de origem criado com sucesso!, Arquivo:", ARQUIVO_ORIGEM)
pedidos

Arquivo de origem criado com sucesso!, Arquivo: data\pedidos_origem.xlsx


,id_pedido,produto,quantidade,valor_total,data_atualizacao
0,1,Hamburguer,2,50.0,2026-11-18 09:00:00
1,2,Pizza,1,40.0,2026-11-18 09:05:00
2,3,Refrigerante,3,30.0,2026-11-18 09:10:00


## 2. Carga Full

In [8]:
df_origem = pd.read_excel(ARQUIVO_ORIGEM)

# A Carga Full vai sobrescrever completamente o destino com os dados da origem
df_destino = df_origem.copy()

# Adicionando uma coluna de data e hora da carga
df_destino["data_carga"] = pd.Timestamp.now()   

df_destino.to_excel(ARQUIVO_DESTINO, index=False)

# Guardar a data da última atualização
ultima_carga = df_destino["data_atualizacao"].max()

print(f"Registros carregados: {len(df_destino)}")
print(f"Última atualização: {ultima_carga}") # Pertinente aos registros
print(f"Última carga realizada em: {pd.Timestamp.now()}") # Pertinente ao proccesso da carga
display(pd.read_excel(ARQUIVO_DESTINO))

Registros carregados: 3
Última atualização: 2026-11-18 09:10:00
Última carga realizada em: 2026-11-18 21:23:40.373535


,id_pedido,produto,quantidade,valor_total,data_atualizacao,data_carga
0,1,Hamburguer,2,50,2026-11-18 09:00:00,2026-11-18 21:23:40.344
1,2,Pizza,1,40,2026-11-18 09:05:00,2026-11-18 21:23:40.344
2,3,Refrigerante,3,30,2026-11-18 09:10:00,2026-11-18 21:23:40.344


## 3. Simular mudanças na Origem

In [9]:
df_origem = pd.read_excel(ARQUIVO_ORIGEM)
df_origem["data_atualizacao"] = pd.to_datetime(df_origem["data_atualizacao"])

# Atualizar o registro do pedido 2
df_origem.loc[df_origem["id_pedido"] == 2, "valor_total"] = 35.00
df_origem.loc[df_origem["id_pedido"] == 2, "data_atualizacao"] = pd.Timestamp("2026-11-18 09:15:00")

novo_pedido = pd.DataFrame([{""
    "id_pedido": 4,
    "produto": "Salada",
    "quantidade": 1,
    "valor_total": 20.00,
    "data_atualizacao": pd.Timestamp("2026-11-18 09:20:00")
    }])


df_origem = pd.concat([df_origem, novo_pedido], ignore_index=True)
df_origem.to_excel(ARQUIVO_ORIGEM, index=False)

display(df_origem)

,id_pedido,produto,quantidade,valor_total,data_atualizacao
0,1,Hamburguer,2,50.0,2026-11-18 09:00:00
1,2,Pizza,1,35.0,2026-11-18 09:15:00
2,3,Refrigerante,3,30.0,2026-11-18 09:10:00
3,4,Salada,1,20.0,2026-11-18 09:20:00


## 4. Carga Incremental

In [10]:
df_origem = pd.read_excel(ARQUIVO_ORIGEM)
df_origem["data_atualizacao"] = pd.to_datetime(df_origem["data_atualizacao"])

# Buscar apenas registros novos ou que foram atualizados desde a última carga
df_incremental = df_origem[df_origem["data_atualizacao"] > ultima_carga].copy()
df_incremental["data_carga"] = pd.Timestamp.now()  # Adicionando a data da carga incremental

print("Registros identificados para carga incremental:", len(df_incremental))
display(df_incremental)

df_destino = pd.read_excel(ARQUIVO_DESTINO)
df_destino["data_atualizacao"] = pd.to_datetime(df_destino["data_atualizacao"])

# Atualizar registros existentes no destino (UPSERT)
df_destino = pd.concat([df_destino, df_incremental], ignore_index=True)
df_destino = (
    df_destino
    .sort_values("data_atualizacao")
    .drop_duplicates(subset="id_pedido", keep="last")
    .sort_values("id_pedido")
)

df_destino.to_excel(ARQUIVO_DESTINO, index=False)

# Atualizar a data e hora da última carga
ultima_carga = df_destino["data_atualizacao"].max()

print(f"Registros carregados: {len(df_destino)}")
print(f"Última atualização: {ultima_carga}") # Pertinente aos registros

display(pd.read_excel(ARQUIVO_DESTINO))

Registros identificados para carga incremental: 2


,id_pedido,produto,quantidade,valor_total,data_atualizacao,data_carga
1,2,Pizza,1,35,2026-11-18 09:15:00,2026-11-18 21:23:40.503301
3,4,Salada,1,20,2026-11-18 09:20:00,2026-11-18 21:23:40.503301


Registros carregados: 4
Última atualização: 2026-11-18 09:20:00


,id_pedido,produto,quantidade,valor_total,data_atualizacao,data_carga
0,1,Hamburguer,2,50,2026-11-18 09:00:00,2026-11-18 21:23:40.344
1,2,Pizza,1,35,2026-11-18 09:15:00,2026-11-18 21:23:40.503
2,3,Refrigerante,3,30,2026-11-18 09:10:00,2026-11-18 21:23:40.344
3,4,Salada,1,20,2026-11-18 09:20:00,2026-11-18 21:23:40.503
